In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader=TextLoader("speech.txt")
documents=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=30)
docs=text_splitter.split_documents(documents)


In [5]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct our

In [9]:
embeddings=OllamaEmbeddings(model="nomic-embed-text:latest")
db=FAISS.from_documents(docs,embeddings)
db

In [10]:
### querying 
query="How does the speaker describe the desired outcome of the war?"
docs=db.similarity_search(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

###  Vector DB As a Retriever

In [13]:
retriever=db.as_retriever()
docs=retriever.invoke(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

### Similarity Search with Score

In [15]:
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='43d48aef-26c6-4434-aed6-30e74e546f8c', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  np.float32(346.45367)),
 (Document(id='3df5916b-1a44-466b-a50c-950da2e7aa94', metadata={'source': 'spe

In [16]:
embedding_vector=embeddings.embed_query(query)
embedding_vector


[-0.3024355173110962,
 0.27900969982147217,
 -3.6137709617614746,
 -0.9991244673728943,
 2.019700527191162,
 1.4686987400054932,
 -1.0226800441741943,
 0.23489181697368622,
 0.48279324173927307,
 -0.17981868982315063,
 -0.06257572025060654,
 0.5964342951774597,
 0.8658819794654846,
 1.6250264644622803,
 2.0871469974517822,
 -0.7816892862319946,
 0.25847128033638,
 -1.208817958831787,
 -0.6822705268859863,
 0.7268785238265991,
 -0.9008129835128784,
 -0.39290162920951843,
 0.09226339310407639,
 0.479070782661438,
 1.3440163135528564,
 0.4108723998069763,
 -0.5249574184417725,
 0.6708530187606812,
 -1.0713884830474854,
 -0.5264062285423279,
 0.9150621294975281,
 0.061426859349012375,
 -0.051089923828840256,
 0.2826700210571289,
 -1.535675287246704,
 -1.9174094200134277,
 0.07922865450382233,
 1.3348684310913086,
 0.35563331842422485,
 -1.4275341033935547,
 0.062406715005636215,
 -0.49671193957328796,
 0.12220105528831482,
 -1.5532352924346924,
 1.3663380146026611,
 -0.6891522407531738,
 -

In [17]:
docs_and_score=db.similarity_search_by_vector(embedding_vector)
docs_and_score

[Document(id='43d48aef-26c6-4434-aed6-30e74e546f8c', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='3df5916b-1a44-466b-a50c-950da2e7aa94', metadata={'source': 'speech.txt'}, page_content='…\n

In [18]:
### Saving and Loading

In [20]:
db.save_local("faiss_index")

In [24]:
new_db=FAISS.load_local("faiss_index",embeddings,allow_dangerous_deserialization=True)

In [27]:
docs=new_db.similarity_search(query)
docs

[Document(id='43d48aef-26c6-4434-aed6-30e74e546f8c', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='3df5916b-1a44-466b-a50c-950da2e7aa94', metadata={'source': 'speech.txt'}, page_content='…\n